In [3]:
import re
import pandas as pd
import pypdf


def extract_pdf_to_dataframe(pdf_path):
    reader = pypdf.PdfReader(pdf_path)
    all_records = []

    pattern = re.compile(
        r"^\s*(.+?)\s+(\d{5})\s+(.+?)\s+(Punjab|Sindh|Khyber Pakhtunkhwa|Balochistan|Islamabad Capital Territory|Islamabad|Azad Kashmir|Gilgit Baltistan)", 
        re.IGNORECASE
    )

    for page in reader.pages:
        text = page.extract_text()
        if not text:
            continue

        for line in text.split("\n"):
            line = line.strip()
            if "GPO" in line:
                match = pattern.search(line)
                if match:
                    office_name = match.group(1).strip()
                    post_code = match.group(2).strip()
                    province = match.group(4).strip()

                    if office_name.upper().endswith("GPO"):
                        all_records.append({
                            "NAME OF DELIVERY POST OFFICE": office_name,
                            "POST CODE": post_code,
                            "PROVINCE": province
                        })

    df_pdf = pd.DataFrame(all_records)
    return df_pdf


def process_and_merge_data(pdf_path, derivation_path, output_path):
    # Step 1 & 2
    gpo_df = extract_pdf_to_dataframe(pdf_path)

    # Step 3: Clean keys
    gpo_df["clean_key"] = gpo_df["NAME OF DELIVERY POST OFFICE"].str.replace(r"\s+GPO\s*$", "", case=False, regex=True)
    gpo_df["clean_key"] = gpo_df["clean_key"].str.strip().str.lower()
    gpo_df["PROVINCE_lower"] = gpo_df["PROVINCE"].str.strip().str.lower()

    # Mappings built from GPO dataset
    district_to_code = dict(zip(gpo_df["clean_key"], gpo_df["POST CODE"]))
    
    # State maps (e.g., Sindh -> Karachi GPO 74200, Punjab -> Lahore GPO 54000)
    state_to_code = gpo_df.groupby("PROVINCE_lower")["POST CODE"].first().to_dict()
    # Islamabad backup handling
    state_to_code["islamabad capital territory"] = "44000"
    state_to_code["islamabad"] = "44000"

    # Load original derivation file
    deriv_df = pd.read_csv(derivation_path)
    
    # Clean verification columns
    deriv_df["district_lower"] = deriv_df["district"].astype(str).str.replace(r"\s+District\s*$", "", case=False, regex=True).str.strip().str.lower()
    deriv_df["state_lower"] = deriv_df["state_di"].astype(str).str.strip().str.lower()

    # --- Step 4: Rule Strict Execution ---
    def fill_logic_strict(row):
        # A. Agar pehle se code mojood hai aur valid hai, to wahi rehne do
        if pd.notna(row["Post_Code_dg"]) and str(row["Post_Code_dg"]).strip() != "" and str(row["Post_Code_dg"]) != "nan":
            return row["Post_Code_dg"]
        
        # B. If the city value is MISSING (isna ya empty string), match using the district
        if pd.isna(row["city"]) or str(row["city"]).strip() == "" or str(row["city"]).lower() == "nan":
            # Check if district is available to match
            if pd.notna(row["district"]) and row["district_lower"] in district_to_code:
                return district_to_code[row["district_lower"]]
                
            # C. If BOTH city AND district are missing, use the state column
            if (pd.isna(row["district"]) or str(row["district"]).strip() == "" or str(row["district"]).lower() == "nan"):
                if row["state_lower"] in state_to_code:
                    return state_to_code[row["state_lower"]]

        # Agar city missing nahi thi par code khaali tha (jaise Islamabad), to direct fallback standard code
        if row["city_lower_temp"] == "islamabad" or row["city_lower_temp"] == "islamabad urban":
            return "44000"

        return row["Post_Code_dg"]

    # Temporary helper for special urban checks
    deriv_df["city_lower_temp"] = deriv_df["city"].astype(str).str.strip().str.lower()

    # Apply strict rules
    deriv_df["Post_Code_dg"] = deriv_df.apply(fill_logic_strict, axis=1)

    # Clean working columns
    deriv_df = deriv_df.drop(columns=["district_lower", "state_lower", "city_lower_temp"])
    
    # Save target file
    deriv_df.to_csv(output_path, index=False)
    print(f"Successfully processed under strict conditions! Output saved: '{output_path}'")


if __name__ == "__main__":
    PDF_FILE = "national post code directory.pdf"
    DERIVATION_FILE = "Pakistan_Post Code  Derivation.csv"
    OUTPUT_FILE = "new_Pakistan_Post Code Derivation.csv"
    
    process_and_merge_data(PDF_FILE, DERIVATION_FILE, OUTPUT_FILE)

Successfully processed under strict conditions! Output saved: 'new_Pakistan_Post Code Derivation.csv'
